In [1]:

from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython import get_ipython
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ip = get_ipython()
if ip is not None:
    ip.run_line_magic("matplotlib", "inline")

ROOT = Path.cwd()
INTERIM = ROOT / "data" / "interim"
REPORTS = ROOT / "reports"
TABLES = REPORTS / "tables"
FIGS = REPORTS / "figures"

if not INTERIM.is_dir():
    raise FileNotFoundError(f"Data directory not found: {INTERIM}")

C = SimpleNamespace(
    ROOT=ROOT,
    INTERIM=INTERIM,
    REPORTS=REPORTS,
    TABLES=TABLES,
    FIGS=FIGS,
    BRAKE_RATES=(0.10, 0.20, 0.30),
    RULE_K=(3, 5, 7, 10),
    F1_FEATURES=["logk", "log_gap", "accel", "log_since_r1"],
    SEED=42,
    HORIZON=5,
    DEEP_M=7,
)

samples_path = INTERIM / "samples.parquet"
m1_clean_path = INTERIM / "m1_clean.parquet"

print(f"Project: {ROOT}")
print(f"Samples: {samples_path} | Exists: {samples_path.is_file()}")
print(f"M1 clean: {m1_clean_path} | Exists: {m1_clean_path.is_file()}")


Project: d:\Academic\UET\ATCS\shock-detector
Samples: d:\Academic\UET\ATCS\shock-detector\data\interim\samples.parquet | Exists: True
M1 clean: d:\Academic\UET\ATCS\shock-detector\data\interim\m1_clean.parquet | Exists: True


# Gan du lieu cua nen M1 da dong gan nhat vao tung event

In [2]:
ONE_MIN = pd.Timedelta(minutes=1)

def decision_bar_key(t: pd.Series, offset=pd.Timedelta(0)) -> pd.Series:
    """Giờ mở của nến M1 mới nhất đã đóng tại thời điểm t (theo đồng hồ M1).."""
    return (pd.to_datetime(t).astype("datetime64[ns]") + offset).dt.floor("min") - ONE_MIN

def attach_last_closed(events: pd.DataFrame, m1: pd.DataFrame, time_col: str = "timestamp",
                       cols=("close",), offset=pd.Timedelta(0), prefix: str = "m1_") -> pd.DataFrame:
    """Gắn vào mỗi sự kiện giá trị của nến M1 mới nhất đã đóng trước t.

    m1: DataFrame có cột `time` giờ mở + các cột cần gắn giá hoặc đặc trưng
        nhân quả đã tính sẵn trên M1.
    Trả về bản sao `events` (giữ thứ tự) thêm:
      {prefix}bar_time : giờ mở nến được dùng
      {prefix}pos      : vị trí dòng của nến đó trong m1
      {prefix}bar_age_min : số phút từ nến đó đến nến lẽ ra phải dùng
                            (0 = liền trước; >0 = đang trong khoảng trống, ví dụ sau cuối tuần)
      {prefix}<col>    : giá trị cột
    """
    m = m1[["time", *cols]].copy()
    m["time"] = m["time"].astype("datetime64[ns]")
    m["_pos"] = np.arange(len(m))
    if not m["time"].is_monotonic_increasing:
        raise ValueError("m1 phải sắp xếp tăng dần theo time")
    ev = events.copy()
    ev["_key"] = decision_bar_key(ev[time_col], offset)
    ev["_order"] = np.arange(len(ev))
    j = pd.merge_asof(ev.sort_values("_key")[["_key", "_order"]], m.rename(columns={"time": "_bar"}),
                      left_on="_key", right_on="_bar", direction="backward", allow_exact_matches=True)
    j = j.sort_values("_order").set_index(ev.index)
    ev[f"{prefix}bar_time"] = j["_bar"]
    ev[f"{prefix}pos"] = j["_pos"].astype("Int64")
    ev[f"{prefix}bar_age_min"] = (j["_key"] - j["_bar"]).dt.total_seconds().div(60)
    for c in cols:
        ev[f"{prefix}{c}"] = j[c]
    return ev.drop(columns=["_key", "_order"])


# Feature engineering

Tính các chỉ số từ dữ liệu giá như ser30, ser60, lrv30 (logarith độ biến động trong 30', có chuẩn hóa), sau đó ghép vào từng sự kiện giao dịch
 
Chọn main feature là volz (đặc trưng đo độ biến động giá cao/thấp bất thường so với mức thông thường) và ser60

In [3]:
"""Đặc trưng M1: họ 2 (volatility) và họ 3 (trend).

Đại diện (mỗi họ 1 biến vào mô hình chính):
  họ 2  vol_z = (log RV30 − mu_h) / sigma_h
        RV30 = sqrt(Σ r²) của 30 return 1 phút; mu_h, sigma_h = mean/std của log RV30 theo giờ server,
        fit trên M1 từ 02 đến HẾT THÁNG TRAIN của từng fold.
  họ 3  ser60 = sign · (C_b − C_{b−60}) / Σ|ΔC| trên 60 phút   (efficiency ratio Kaufman, có dấu)
        ∈ [−1, 1]; < 0 = giá đi ngược basket; |ser60| = ER60 (độ "thẳng" của đường giá).
  Biến chỉ dùng cho độ nhạy: ser30, ret60_z = sign · Σr / sqrt(Σr²) trên 60 phút.

"""

import numpy as np
import pandas as pd


RV_WIN = 30
TREND_WIN = 60
TREND_WIN_SHORT = 30
BAR_COLS = ("lrv30", "ser60", "ser30", "ret60_z")
MAIN_FEATURES = {"vol": ["vol_z"], "trend": ["ser60"]}


def _contig_run(gap_min: np.ndarray) -> np.ndarray:
    """Số return 1 phút liên tục kết thúc tại mỗi nến (0 ở nến đầu / ngay sau khoảng trống)."""
    ok = np.nan_to_num(gap_min, nan=0.0) == 1
    run = np.zeros(len(ok), dtype=np.int64)
    c = 0
    for i, v in enumerate(ok):
        c = c + 1 if v else 0
        run[i] = c
    return run


def _win_sum(x: np.ndarray, n: int) -> np.ndarray:
    """Tổng n phần tử kết thúc tại i (gồm i): cs[i] − cs[i−n]. NaN khi i < n."""
    cs = np.concatenate([[0.0], np.cumsum(x)])
    out = np.full(len(x), np.nan)
    out[n - 1:] = cs[n:] - cs[:-n]
    return out


def bar_features(m1: pd.DataFrame) -> pd.DataFrame:
    """Đặc trưng theo nến, chỉ dùng nến ≤ chính nó. m1 có time, close, gap_min, sắp xếp tăng dần.
    ser*, ret60_z chưa nhân dấu basket (dấu thị trường: > 0 = giá tăng)."""
    t = m1["time"].astype("datetime64[ns]")
    if not t.is_monotonic_increasing:
        raise ValueError("m1 phải sắp xếp tăng dần theo time")
    c = m1["close"].to_numpy(float)
    lc = np.log(c)
    run = _contig_run(m1["gap_min"].to_numpy(float))
    r = np.diff(lc, prepend=np.nan)
    dc = np.diff(c, prepend=np.nan)
    bad = run == 0
    r = np.where(bad, 0.0, r)
    dc = np.where(bad, 0.0, dc)

    def valid(n):
        return run >= n

    out = pd.DataFrame({"time": t.to_numpy()})
    rv30 = np.sqrt(_win_sum(r ** 2, RV_WIN))
    out["lrv30"] = np.where(valid(RV_WIN) & (rv30 > 0), np.log(np.where(rv30 > 0, rv30, 1.0)), np.nan)
    for n, name in ((TREND_WIN, "ser60"), (TREND_WIN_SHORT, "ser30")):
        net = _win_sum(dc, n)
        path = _win_sum(np.abs(dc), n)
        out[name] = np.where(valid(n) & (path > 0), net / np.where(path > 0, path, 1.0), np.nan)
    s1, s2 = _win_sum(r, TREND_WIN), _win_sum(r ** 2, TREND_WIN)
    out["ret60_z"] = np.where(valid(TREND_WIN) & (s2 > 0), s1 / np.sqrt(np.where(s2 > 0, s2, 1.0)), np.nan)
    out["hour"] = t.dt.hour.to_numpy()
    return out


def fit_hour_norm(bars: pd.DataFrame, cutoff) -> pd.DataFrame:
    """mu_h, sigma_h của lrv30 theo giờ server, chỉ trên nến có time < cutoff."""
    b = bars[(bars["time"] < pd.Timestamp(cutoff)) & bars["lrv30"].notna()]
    return b.groupby("hour")["lrv30"].agg(mu="mean", sd="std", n="size")


def apply_hour_norm(lrv: pd.Series, hour: pd.Series, norm: pd.DataFrame) -> pd.Series:
    mu = hour.map(norm["mu"])
    sd = hour.map(norm["sd"])
    return (lrv - mu) / sd

def attach_features(samples: pd.DataFrame, bars: pd.DataFrame, offset=pd.Timedelta(0)) -> pd.DataFrame:
    """Gắn đặc trưng của nến đã đóng gần nhất; nhân dấu basket (BUY +1, SELL −1).
    Trả bản sao samples + m1_bar_time, m1_bar_age_min, hour, lrv30, ser60, ser30, ret60_z, m1_ok."""
    ev = attach_last_closed(samples, bars, time_col="timestamp", cols=(*BAR_COLS, "hour"),
                            offset=offset, prefix="m1_")
    sign = np.where(ev["side"].astype(str).str.upper() == "BUY", 1.0, -1.0)
    out = samples.copy()
    out["m1_bar_time"] = ev["m1_bar_time"]
    out["m1_bar_age_min"] = ev["m1_bar_age_min"]
    out["hour"] = ev["m1_hour"]
    out["lrv30"] = ev["m1_lrv30"]
    for c in ("ser60", "ser30", "ret60_z"):
        out[c] = ev[f"m1_{c}"] * sign
    out.loc[out["m1_bar_age_min"] != 0, ["lrv30", "ser60", "ser30", "ret60_z"]] = np.nan
    out["m1_ok"] = out[["lrv30", "ser60", "ser30", "ret60_z"]].notna().all(axis=1)
    return out


def with_fold_vol(samples: pd.DataFrame, bars: pd.DataFrame, test_month: str):
    """Bản sao samples có vol_z chuẩn hóa bằng mu_h, sigma_h fit trên M1 trước tháng test."""
    norm = fit_hour_norm(bars, pd.Timestamp(test_month + "-01"))
    s = samples.copy()
    s["vol_z"] = apply_hour_norm(s["lrv30"], s["hour"], norm)
    return s, norm


def check_decision_bar(feat: pd.DataFrame) -> int:
    ok = feat["m1_bar_time"].isna() | (feat["m1_bar_time"] + pd.Timedelta(minutes=1) <= feat["timestamp"])
    return int((~ok).sum())


def check_truncation(m1: pd.DataFrame, cuts) -> float:
    """Đặc trưng nến <= cut tính trên M1 bị cắt tại cut phải trùng bản đầy đủ. Trả max |chênh|."""
    full = bar_features(m1)
    worst = 0.0
    for cut in cuts:
        part = bar_features(m1[m1["time"] <= cut].reset_index(drop=True))
        a = full.loc[full["time"] <= cut, BAR_COLS].to_numpy()
        b = part[list(BAR_COLS)].to_numpy()
        if not np.array_equal(np.isnan(a), np.isnan(b)):
            return np.inf
        d = np.nanmax(np.abs(a - b)) if np.isfinite(a).any() else 0.0
        worst = max(worst, float(d))
    return worst


def check_perturb_future(samples: pd.DataFrame, m1: pd.DataFrame, events_idx, seed=0) -> tuple[int, int]:
    """Với mỗi sự kiện: làm nhiễu MỌI nến có giờ mở ≥ floor(t) (nến đang chạy + tương lai),
    tính lại đặc trưng. Trả (số sự kiện đặc trưng bị đổi — phải 0, số sự kiện kế tiếp bị đổi — nên > 0)."""
    rng = np.random.default_rng(seed)
    cols = ["lrv30", "ser60", "ser30", "ret60_z"]
    base = attach_features(samples, bar_features(m1))
    changed, sensitive = 0, 0
    for i in events_idx:
        t_floor = samples.loc[i, "timestamp"].floor("min")
        m = m1.copy()
        fut = m["time"] >= t_floor
        m.loc[fut, "close"] = m.loc[fut, "close"] * np.exp(rng.normal(0, 2e-3, int(fut.sum())))
        new = attach_features(samples, bar_features(m))
        a, b = base.loc[i, cols].to_numpy(float), new.loc[i, cols].to_numpy(float)
        if not np.allclose(a, b, equal_nan=True, rtol=0, atol=1e-12):
            changed += 1
        later = samples.index[(samples["timestamp"] >= t_floor + pd.Timedelta(minutes=3))
                              & base["m1_ok"]]
        if len(later):
            j = later[np.argmin(samples.loc[later, "timestamp"].to_numpy())]
            if not np.allclose(base.loc[j, cols].to_numpy(float), new.loc[j, cols].to_numpy(float),
                               equal_nan=True, rtol=0, atol=1e-12):
                sensitive += 1
    return changed, sensitive


def check_norm_no_test(bars: pd.DataFrame, cutoff, seed=0) -> float:
    """mu_h, sigma_h không đổi khi làm nhiễu lrv30 của nến >= cutoff. Trả max |chênh|."""
    a = fit_hour_norm(bars, cutoff)
    b2 = bars.copy()
    m = b2["time"] >= pd.Timestamp(cutoff)
    b2.loc[m, "lrv30"] = b2.loc[m, "lrv30"] + np.random.default_rng(seed).normal(0, 5, int(m.sum()))
    b = fit_hour_norm(b2, cutoff)
    return float(np.nanmax(np.abs(a[["mu", "sd"]].to_numpy() - b[["mu", "sd"]].to_numpy())))


def steps_last_60m(samples: pd.DataFrame) -> pd.Series:
    """Số bậc của chính basket mở trong (t − 60 phút, t], gồm Rk hiện tại và R1 nếu trong cửa sổ."""
    out = pd.Series(0, index=samples.index, dtype="int64")
    w = np.int64(60 * 60 * 10**9)
    for _, d in samples.groupby("basket_id", sort=False):
        ts = d["timestamp"].to_numpy().astype("datetime64[ns]").astype("int64")
        o = np.argsort(ts)
        tss = ts[o]
        cnt = np.searchsorted(tss, tss, "right") - np.searchsorted(tss, tss - w, "right")
        since_r1 = np.expm1(d["log_since_r1"].to_numpy()[o])     # phút
        cnt = cnt + (since_r1 < 60)
        out.loc[d.index[o]] = cnt
    return out


def walk_forward_m1(samples: pd.DataFrame, bars: pd.DataFrame, test_months, features, **kw):
    """Gọi evaluate.walk_forward từng tháng một; trước mỗi fold tính lại vol_z bằng mu_h, sigma_h
    fit trên M1 < đầu tháng test. Train và test của fold dùng cùng một chuẩn hóa.
    Thứ tự dòng đầu ra giống walk_forward(samples, test_months, ...)."""
    outs = []
    for m in test_months:
        s = with_fold_vol(samples, bars, m)[0] if "vol_z" in (features or []) else samples
        if features is not None and s[features].isna().any().any():
            raise ValueError(f"NaN trong đặc trưng ở fold {m}; lọc dòng trước khi gọi")
        outs.append(walk_forward(s, [m], features=features, **kw))
    return pd.concat(outs, ignore_index=True)

F = SimpleNamespace(**{'RV_WIN': RV_WIN, 'TREND_WIN': TREND_WIN, 'TREND_WIN_SHORT': TREND_WIN_SHORT, 'BAR_COLS': BAR_COLS, 'MAIN_FEATURES': MAIN_FEATURES, '_contig_run': _contig_run, '_win_sum': _win_sum, 'bar_features': bar_features, 'fit_hour_norm': fit_hour_norm, 'apply_hour_norm': apply_hour_norm, 'attach_features': attach_features, 'with_fold_vol': with_fold_vol, 'check_decision_bar': check_decision_bar, 'check_truncation': check_truncation, 'check_perturb_future': check_perturb_future, 'check_norm_no_test': check_norm_no_test, 'steps_last_60m': steps_last_60m, 'walk_forward_m1': walk_forward_m1})


In [4]:
"""
Feature engineering bổ sung, với
- c1: ser240_s: xu hướng giá trong 240', có xét hướng. giá đi thuận basket -> dương, ngược là âm 
- c2: exc: mức độ giá đi ngược hướng basket từ R1, chuẩn hóa theo biên độ giá trung bình 60 nến. giá đi ngược càng nhiều exc càng lớn
- c3: kcach từ giá htai đến đáy 4 giờ (BUY) hoặc đỉnh (sell), chuẩn hóa theo biên độ giá. càng gần biên càng bất lợi -> brk càng nhỏ

nhằm mục đích bổ sung thông tin về xu hướng (trend), mức bất lợi của giá và nguy cơ phá vùng giá bất lợi
"""

import numpy as np
import pandas as pd


W = 240
ATR_N = 60
SEG_GAP_MIN = 360
BAR_X = ("close", "atr60", "ser240", "lo240", "hi240")


def extra_bar_features(m1: pd.DataFrame, w: int = W, atr_n: int = ATR_N, seg_gap_min: float = SEG_GAP_MIN) -> pd.DataFrame:
    t = m1["time"].astype("datetime64[ns]")
    if not t.is_monotonic_increasing:
        raise ValueError("m1 phải sắp xếp tăng dần theo time")
    c = m1["close"].to_numpy(float)
    gap = m1["gap_min"].to_numpy(float)
    new_seg = np.isnan(gap) | (gap > seg_gap_min)
    seg = np.cumsum(new_seg)
    pos = pd.Series(np.arange(len(m1))).groupby(seg).cumcount().to_numpy()
    dc = np.where(new_seg, 0.0, np.diff(c, prepend=np.nan))
    cs = np.concatenate([[0.0], np.cumsum(np.abs(dc))])
    path = np.full(len(c), np.nan)
    path[w:] = cs[w + 1:] - cs[1:len(c) - w + 1]
    net = np.full(len(c), np.nan)
    net[w:] = c[w:] - c[:-w]
    ok = pos >= w
    ser = np.where(ok & (path > 0), net / np.where(path > 0, path, 1.0), np.nan)
    g = pd.DataFrame({"h": m1["high"].to_numpy(float), "l": m1["low"].to_numpy(float), "seg": seg})
    rng = g["h"] - g["l"]
    atr = rng.groupby(g["seg"]).transform(lambda s: s.rolling(atr_n, min_periods=atr_n).mean())
    lo = g["l"].groupby(g["seg"]).transform(lambda s: s.rolling(w + 1, min_periods=w + 1).min())
    hi = g["h"].groupby(g["seg"]).transform(lambda s: s.rolling(w + 1, min_periods=w + 1).max())
    return pd.DataFrame({"time": t.to_numpy(), "close": c, "atr60": atr.to_numpy(), "ser240": ser,
                         "lo240": lo.to_numpy(), "hi240": hi.to_numpy()})


def attach_extra(samples: pd.DataFrame, xbars: pd.DataFrame) -> pd.DataFrame:
    """Trả DataFrame (index = samples.index): ser240_s, exc, brk, x_ok. Chỉ dùng nến đã đóng trước t
    (C3, C1) và nến đã đóng trước R1 (C2)."""
    sign = np.where(samples["side"].astype(str).str.upper() == "BUY", 1.0, -1.0)
    now = attach_last_closed(samples, xbars, time_col="timestamp", cols=BAR_X, prefix="x_")
    ev1 = samples[["timestamp", "log_since_r1"]].copy()
    ev1["timestamp"] = samples["timestamp"] - pd.to_timedelta(np.expm1(samples["log_since_r1"].to_numpy()), unit="m")
    r1 = attach_last_closed(ev1, xbars, time_col="timestamp", cols=("close",), prefix="r1_")
    bad = (now["x_bar_age_min"] != 0) | (r1["r1_bar_age_min"] != 0)
    atr = now["x_atr60"]
    out = pd.DataFrame(index=samples.index)
    out["ser240_s"] = now["x_ser240"] * sign
    out["exc"] = np.arcsinh(sign * (r1["r1_close"] - now["x_close"]) / atr)
    d = np.where(sign > 0, now["x_close"] - now["x_lo240"], now["x_hi240"] - now["x_close"])
    out["brk"] = np.log1p(np.clip(d, 0, None) / atr)
    out.loc[bad, ["ser240_s", "exc", "brk"]] = np.nan
    out["x_bar_time"] = now["x_bar_time"]
    out["x_ok"] = out[["ser240_s", "exc", "brk"]].notna().all(axis=1)
    return out

X = SimpleNamespace(**{'W': W, 'ATR_N': ATR_N, 'SEG_GAP_MIN': SEG_GAP_MIN, 'BAR_X': BAR_X, 'extra_bar_features': extra_bar_features, 'attach_extra': attach_extra})


Khung đánh giá: walk-forward theo tháng, PR-AUC, bootstrap theo ngày, mô phỏng phanh

In [5]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def make_fold(samples, month):
    """Train = dòng của các tháng trước, loại basket kết thúc sau mốc cắt.
    Test = dòng thuộc tháng `month`."""
    ms = pd.Timestamp(month + "-01")
    train = samples[samples["t_end"] < ms]
    test = samples[samples["month"] == month]
    n_purged = int(((samples["timestamp"] < ms) & (samples["t_end"] >= ms)).sum())
    return train, test, n_purged


def walk_forward(samples, test_months, features=None, rates=C.BRAKE_RATES, C_reg=1.0):
    """features=None -> luật theo độ sâu score=k. Ngược lại logistic L2.
    Ngưỡng cho từng tỷ lệ phanh được chọn trên train."""
    outs = []
    for m in test_months:
        tr, te, _ = make_fold(samples, m)
        if len(te) == 0 or len(tr) == 0:
            continue
        if features is None:
            s_tr, s_te = tr["k"].to_numpy(float), te["k"].to_numpy(float)
        else:
            pipe = make_pipeline(StandardScaler(), LogisticRegression(C=C_reg, max_iter=1000))
            pipe.fit(tr[features], tr["y"])
            s_tr = pipe.predict_proba(tr[features])[:, 1]
            s_te = pipe.predict_proba(te[features])[:, 1]
        o = te[["basket_id", "k", "y", "M", "month", "day"]].copy()
        o["score"] = s_te
        for q in rates:
            o[f"thr_{round(q * 100)}"] = np.quantile(s_tr, 1 - q)
        o["fold"] = m
        outs.append(o)
    return pd.concat(outs, ignore_index=True)


def pr_auc(y, s):
    y = np.asarray(y)
    return float("nan") if y.sum() == 0 else float(average_precision_score(y, s))


def paired_bootstrap(y, s_a, s_b, groups, n_boot=1000, seed=C.SEED):
    """Bootstrap theo cụm (ngày), ghép cặp hai mô hình trên cùng mẫu lặp."""
    y, s_a, s_b = map(np.asarray, (y, s_a, s_b))
    codes, uniq = pd.factorize(np.asarray(groups))
    idx_by = [np.flatnonzero(codes == i) for i in range(len(uniq))]
    rng = np.random.default_rng(seed)
    a, b = [], []
    for _ in range(n_boot):
        pick = rng.integers(0, len(uniq), len(uniq))
        idx = np.concatenate([idx_by[i] for i in pick])
        if y[idx].sum() == 0:
            continue
        a.append(average_precision_score(y[idx], s_a[idx]))
        b.append(average_precision_score(y[idx], s_b[idx]))
    a, b = np.array(a), np.array(b)
    ci = lambda x: (float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5)))
    return dict(ap_a=pr_auc(y, s_a), ci_a=ci(a), ap_b=pr_auc(y, s_b), ci_b=ci(b),
                delta=pr_auc(y, s_a) - pr_auc(y, s_b), ci_delta=ci(a - b),
                frac_delta_pos=float((a - b > 0).mean()), n_boot=len(a))


def brake_sim(oos, thr):
    # giả lập xem hệ thống quyết định dừng mở thêm basket thì xảy ra gì
    d = oos[["basket_id", "k", "y", "M", "score"]].copy()
    d["flag"] = d["score"].to_numpy() >= np.asarray(thr)
    b = d.groupby("basket_id", sort=False).agg(M=("M", "first"))
    first = (d[d["flag"]].sort_values(["basket_id", "k"]).groupby("basket_id", sort=False)
             .first()[["k", "y"]].rename(columns={"k": "k_halt", "y": "y_halt"}))
    b = b.join(first)
    halted = b["k_halt"].notna()
    deep = b["M"] >= C.DEEP_M
    avoided = (b["M"] - b["k_halt"] + 1).where(halted, 0)
    n_b, n_deep = len(b), int(deep.sum())
    caught = int(((b["y_halt"] == 1) & deep).sum())
    false_h = int((b["y_halt"] == 0).sum())
    return dict(
        brake_rate_rows=float(d["flag"].mean()),
        n_baskets=n_b, n_deep=n_deep,
        halted_pct=float(halted.mean()),
        deep_caught=caught,
        deep_caught_pct=caught / n_deep if n_deep else float("nan"),
        avoided_total=int(avoided.sum()),
        avoided_pct=float(avoided.sum() / max((b["M"] - 1).sum(), 1)),
        avoided_deep=int(avoided[deep].sum()),
        false_flag_rows=int((d["flag"] & (d["y"] == 0)).sum()),
        false_halts=false_h,
        false_halts_per100=100 * false_h / n_b,
    )



Config các thang đánh giá, feature và bootstrap để so sánh

In [6]:
TM = ["2026-04", "2026-05", "2026-06", "2026-07", "2026-08"]
F1 = list(C.F1_FEATURES)
A3 = F1 + ["vol_z", "ser60"]
NEW = ["ser240_s", "exc", "brk"]
N_BOOT = 1000
N_PRIMARY = 4

class Tee:
    def __init__(self, p):
        self.f = open(p, "w", encoding="utf-8")

    def __call__(self, *a):
        s = " ".join(str(x) for x in a)
        print(s, flush=True)
        self.f.write(s + "\n")
        self.f.flush()

def boot(y, a, b, groups, n_boot=N_BOOT, seed=C.SEED):
    """Bootstrap theo ngày, ghép cặp; trả delta, CI 95% và CI hiệu chỉnh Bonferroni (1 − 0,05/4)."""
    y, a, b = map(np.asarray, (y, a, b))
    codes, uniq = pd.factorize(np.asarray(groups))
    idx_by = [np.flatnonzero(codes == i) for i in range(len(uniq))]
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(n_boot):
        idx = np.concatenate([idx_by[i] for i in rng.integers(0, len(uniq), len(uniq))])
        if y[idx].sum() == 0:
            continue
        d.append(average_precision_score(y[idx], a[idx]) - average_precision_score(y[idx], b[idx]))
    d = np.array(d)
    al = 0.05 / N_PRIMARY
    return dict(delta=pr_auc(y, a) - pr_auc(y, b), lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5),
                lo_b=np.percentile(d, 100 * al / 2), hi_b=np.percentile(d, 100 * (1 - al / 2)), p_pos=float((d > 0).mean()))



Đánh giá 3 features mới ser240_s, exc, brk có cải thiện không
với

- ser240s: efficiency rate trên 240 nến M1 có dấu
- exc: Quãng giá đi ngược hướng basket kể từ R1, chuẩn hóa theo ATR60
- brk: khoảng cách đến đỉnh/đáy 4h theo hướng bất lợi, chuẩn hóa theo atr60 và biến đổi log1p

thử nghiệm với baseline đơn giản là A0 = log(k), baseline chính là F1 + volz + ser60, mô hình chính là baseline chính + từng feature trên

In [7]:
for d in (C.REPORTS, C.TABLES, C.FIGS):
    Path(d).mkdir(parents=True, exist_ok=True)
log = Tee(C.REPORTS / "g5x_new_report.txt")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
t0 = time.time()
log("G5 KHÁM PHÁ — 3 ứng viên ngoài 5 họ (chạy một lần, chốt trước)")
log("C1 ser240_s: efficiency ratio 240 nến M1 có dấu | C2 exc: asinh(quãng giá ngược basket từ R1 / ATR60) | "
    "C3 brk: log1p(khoảng cách tới đỉnh/đáy 4 giờ ngược basket / ATR60)")
log(f"W = {X.W} nến, ATR = {X.ATR_N} nến, phân đoạn tại gap > {X.SEG_GAP_MIN} phút; n_boot = {N_BOOT}; "
    f"so sánh chính = {N_PRIMARY} (Bonferroni α = 0,05/{N_PRIMARY} → CI {100 * (1 - 0.05 / N_PRIMARY):.2f}%)")

s = pd.read_parquet(C.INTERIM / "samples.parquet")
m1 = pd.read_parquet(C.INTERIM / "m1_clean.parquet")
m1["time"] = m1["time"].astype("datetime64[ns]")
bars = F.bar_features(m1)
feat = F.attach_features(s, bars)
xb = X.extra_bar_features(m1)
ex = X.attach_extra(s, xb)

log("\n" + "=" * 70 + "\n## A. DÒNG BỊ LOẠI\n" + "=" * 70)
g4 = feat["m1_ok"]
drop = g4 & ~ex["x_ok"]
log(f"Tập G4/G5: {int(g4.sum()):,} dòng. Loại thêm do cửa sổ 4 giờ/ATR: {int(drop.sum())} ({100 * drop.sum() / g4.sum():.2f}%) "
    f"→ vượt ngưỡng 1%: đánh giá mọi mô hình trên cùng tập, A3 tính lại trên tập này.")
dd = s[drop]
log(f"  theo giờ của t: {dd['timestamp'].dt.hour.value_counts().sort_index().to_dict()} (4 giờ đầu phiên sau cuối tuần)")
log(f"  theo tháng: {dd['month'].value_counts().sort_index().to_dict()} | dương bị loại: {int(dd['y'].sum())} "
    f"(prevalence phần bị loại {dd['y'].mean():.3f} so với {s.loc[g4, 'y'].mean():.3f})")
S = feat[g4 & ex["x_ok"]].join(ex[NEW]).copy()
log(f"Tập chung G5-khám phá: {len(S):,} dòng, dương {int(S['y'].sum())}")


log("\n" + "=" * 70 + "\n## B. KIỂM TRA CHỐNG NHÌN TƯƠNG LAI\n" + "=" * 70)
rng = np.random.default_rng(C.SEED)
checks = []
used = S["timestamp"].dt.floor("min") - ex.loc[S.index, "x_bar_time"]
checks.append(("B1 nến dùng = floor(t) − 1 phút", bool((used == pd.Timedelta(minutes=1)).all()),
               f"giá trị = {sorted(used.dt.total_seconds().div(60).unique().tolist())}"))

worst = 0.0
full = xb.set_index("time")
for cut in pd.to_datetime(rng.choice(m1["time"].to_numpy()[8000:], 8, replace=False)):
    part = X.extra_bar_features(m1[m1["time"] <= cut].reset_index(drop=True)).set_index("time")
    a = full.loc[part.index, ["atr60", "ser240", "lo240", "hi240"]].to_numpy()
    b = part[["atr60", "ser240", "lo240", "hi240"]].to_numpy()
    if not np.array_equal(np.isnan(a), np.isnan(b)):
        worst = np.inf
    elif np.isfinite(a).any():
        worst = max(worst, float(np.nanmax(np.abs(a - b))))
checks.append(("B2 cắt M1 tại 8 mốc: đặc trưng nến ≤ mốc không đổi", worst == 0, f"max |chênh| = {worst:g}"))

pool = rng.choice(S.index.to_numpy(), 12, replace=False)
changed, sens = 0, 0
for i in pool:
    tf = s.loc[i, "timestamp"].floor("min")
    mp = m1.copy()
    fut = mp["time"] >= tf
    f = np.exp(rng.normal(0, 2e-3, int(fut.sum())))
    for col in ("open", "high", "low", "close"):
        mp.loc[fut, col] = mp.loc[fut, col].to_numpy() * f
    new = X.attach_extra(s.loc[[i]], X.extra_bar_features(mp))[NEW].to_numpy(float)
    if not np.allclose(ex.loc[[i], NEW].to_numpy(float), new, rtol=0, atol=1e-12, equal_nan=True):
        changed += 1
    j = S.index[(S["timestamp"] >= tf + pd.Timedelta(minutes=10)) & (S["timestamp"] < tf + pd.Timedelta(hours=6))]
    if len(j):
        sens += int(not np.allclose(ex.loc[[j[0]], NEW].to_numpy(float),
                                    X.attach_extra(s.loc[[j[0]]], X.extra_bar_features(mp))[NEW].to_numpy(float),
                                    rtol=0, atol=1e-12, equal_nan=True))
checks.append(("B3 nhiễu mọi nến ≥ floor(t) (OHLC), 12 sự kiện: đặc trưng tại t không đổi", changed == 0,
               f"bị đổi = {changed}"))
checks.append(("B4 độ nhạy: sự kiện kế tiếp (t+10' đến t+6h) có đổi", sens > 0, f"{sens}/12"))

r1t = s["timestamp"] - pd.to_timedelta(np.expm1(s["log_since_r1"].to_numpy()), unit="m")
checks.append(("B5 thời điểm R1 ≤ t (giá R1 là quá khứ)", bool((r1t <= s["timestamp"]).all()), ""))
for n, ok, note in checks:
    log(f"  [{'OK' if ok else '!!'}] {n}  {note}")


log("\n" + "=" * 70 + "\n## C. ABLATION (PR-AUC gộp OOS 04–08, cùng tập dòng)\n" + "=" * 70)
MODELS = {
    "A0 log(k)": ["logk"],
    "A3 (mốc)": A3,
    "A3 + C1 ser240": A3 + ["ser240_s"],
    "A3 + C2 exc": A3 + ["exc"],
    "A3 + C3 brk": A3 + ["brk"],
    "A3 + C1+C2+C3": A3 + NEW,
    "log(k)+ser60": ["logk", "ser60"],
    "log(k) + C1": ["logk", "ser240_s"],
    "log(k) + C2": ["logk", "exc"],
    "log(k) + C3": ["logk", "brk"],
}
oos = {n: F.walk_forward_m1(S, bars, TM, fs) for n, fs in MODELS.items()}
ref = oos["A0 log(k)"]
for n, o in oos.items():
    assert (o["basket_id"].to_numpy() == ref["basket_id"].to_numpy()).all() and (o["k"].to_numpy() == ref["k"].to_numpy()).all()
log(f"OOS: {len(ref):,} dòng, dương {int(ref['y'].sum())}, prevalence {ref['y'].mean():.4f}, {ref['day'].nunique()} ngày")
rows = [{"model": n, "ap": pr_auc(o["y"], o["score"])} for n, o in oos.items()]
ap = pd.DataFrame(rows)
log(ap.round(4).to_string(index=False))

pairs = [("A3 + C1 ser240", "A3 (mốc)", "chính"), ("A3 + C2 exc", "A3 (mốc)", "chính"),
         ("A3 + C3 brk", "A3 (mốc)", "chính"), ("A3 + C1+C2+C3", "A3 (mốc)", "chính"),
         ("log(k) + C1", "A0 log(k)", "phụ"), ("log(k) + C2", "A0 log(k)", "phụ"),
         ("log(k) + C3", "A0 log(k)", "phụ"), ("log(k)+ser60", "A0 log(k)", "đối chiếu"),
         ("A3 (mốc)", "A0 log(k)", "đối chiếu")]
rows = []
for a, b, kind in pairs:
    r = boot(oos[a]["y"], oos[a]["score"], oos[b]["score"], oos[a]["day"])
    rows.append({"loai": kind, "so_sanh": f"{a} − {b}", "delta": r["delta"], "ci95_lo": r["lo"], "ci95_hi": r["hi"],
                 "ciBonf_lo": r["lo_b"], "ciBonf_hi": r["hi_b"], "P(delta>0)": r["p_pos"]})
dt = pd.DataFrame(rows)
dt.to_csv(C.TABLES / "g5x_new_delta.csv", index=False)
log("\nΔPR-AUC ghép cặp (bootstrap theo ngày). ciBonf = CI hiệu chỉnh cho 4 so sánh chính.")
log(dt.round(4).to_string(index=False))
ap.to_csv(C.TABLES / "g5x_new_ap.csv", index=False)

bym = pd.DataFrame({n: o.groupby("month").apply(lambda d: pr_auc(d["y"], d["score"]), include_groups=False)
                    for n, o in oos.items() if n in ("A0 log(k)", "A3 (mốc)", "A3 + C1 ser240", "A3 + C2 exc",
                                                     "A3 + C3 brk", "A3 + C1+C2+C3")})
bym.to_csv(C.TABLES / "g5x_new_by_month.csv")
log("\nPR-AUC theo tháng:")
log(bym.round(4).to_string())
for n in ("A3 + C1 ser240", "A3 + C2 exc", "A3 + C3 brk", "A3 + C1+C2+C3"):
    log(f"  số tháng '{n}' > A3: {int((bym[n] > bym['A3 (mốc)']).sum())}/5")


log("\n" + "=" * 70 + "\n## D. PHÂN TẦNG THEO k (so với A3, CI 95%)\n" + "=" * 70)
rows = []
for n in ("A3 + C1 ser240", "A3 + C2 exc", "A3 + C3 brk", "A3 + C1+C2+C3"):
    for lo, hi, lab in [(2, 4, "2-4"), (5, 9, "5-9"), (10, 999, ">=10")]:
        m = ((ref["k"] >= lo) & (ref["k"] <= hi)).to_numpy()
        r = boot(oos[n]["y"][m], oos[n]["score"][m], oos["A3 (mốc)"]["score"][m], ref["day"][m])
        rows.append({"model": n, "k": lab, "n": int(m.sum()), "delta": r["delta"], "ci_lo": r["lo"], "ci_hi": r["hi"]})
st = pd.DataFrame(rows)
st.to_csv(C.TABLES / "g5x_new_by_k.csv", index=False)
log(st.round(4).to_string(index=False))


log("\n" + "=" * 70 + "\n## E. HỆ SỐ CHUẨN HÓA (A3 + C1+C2+C3, từng fold) VÀ TƯƠNG QUAN\n" + "=" * 70)
fs = MODELS["A3 + C1+C2+C3"]
coefs = {}
for m in TM:
    Sm = F.with_fold_vol(S, bars, m)[0]
    tr, _, _ = make_fold(Sm, m)
    pipe = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000)).fit(tr[fs], tr["y"])
    coefs[m] = pd.Series(pipe[-1].coef_[0], index=fs)
ct = pd.DataFrame(coefs)
ct["mean"] = ct.mean(axis=1)
ct["sign_stable"] = np.sign(ct[TM]).nunique(axis=1) == 1
ct.to_csv(C.TABLES / "g5x_new_coef.csv")
log(ct.round(3).to_string())
log("\nSpearman (tập chung):")
log(S[["logk", "ser60", "ser240_s", "exc", "brk", "vol_z" if "vol_z" in S else "lrv30"]].corr("spearman").round(2).to_string())


log("\n" + "=" * 70 + "\n## F. MÔ PHỎNG PHANH @10/20/30% — CẬN TRÊN\n" + "=" * 70)
rows = []
for n in ("A0 log(k)", "A3 (mốc)", "A3 + C1 ser240", "A3 + C2 exc", "A3 + C3 brk", "A3 + C1+C2+C3"):
    o = oos[n]
    for q in C.BRAKE_RATES:
        r = brake_sim(o, o[f"thr_{round(q * 100)}"])
        rows.append({"model": n, "target": q, "brake_rows_pct": 100 * r["brake_rate_rows"], "deep_caught": r["deep_caught"],
                     "avoided_total": r["avoided_total"], "false_halts": r["false_halts"],
                     "false_per100": r["false_halts_per100"]})
bt = pd.DataFrame(rows)
bt.to_csv(C.TABLES / "g5x_new_brake.csv", index=False)
log(bt.round(2).to_string(index=False))


fig, ax = plt.subplots(figsize=(7.5, 3.4))
d = dt[dt["loai"] == "chính"].reset_index(drop=True)
yy = np.arange(len(d))
ax.errorbar(d["delta"], yy, xerr=[d["delta"] - d["ciBonf_lo"], d["ciBonf_hi"] - d["delta"]], fmt="none", ecolor="C1", capsize=2,
            lw=1, label="CI hiệu chỉnh (98,75%)")
ax.errorbar(d["delta"], yy, xerr=[d["delta"] - d["ci95_lo"], d["ci95_hi"] - d["delta"]], fmt="o", color="C0", capsize=3,
            label="CI 95%")
ax.axvline(0, color="grey", lw=0.8)
ax.set_yticks(yy, [x.split(" − ")[0] for x in d["so_sanh"]])
ax.invert_yaxis()
ax.set_xlabel("ΔPR-AUC so với A3 (cùng tập dòng)")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(C.FIGS / "g5x_new_delta.png", dpi=130)
display(fig)
plt.close(fig)
out = S[["basket_id", "k", "timestamp"]].join(S[NEW])
out.to_parquet(C.INTERIM / "g5x_new_features.parquet", index=False)
log(f"\nTổng thời gian {time.time() - t0:.0f} s")
log(f"Kiểm tra rò rỉ: {'tất cả OK' if all(ok for _, ok, _ in checks) else '!! CÓ LỖI'}")

NameError: name 'time' is not defined